In [2]:
import pandas as pd
import h5py
import numpy as np


In [134]:
def restructure_features(input_file, output_file):
    #step 1
    with h5py.File(input_file, 'r') as f:
        with h5py.File(output_file, 'w') as new_file:
            for name, obj in f.items():
                new_group = new_file.create_group(name)
                for n2, o2 in obj.items():
                    if n2 != 'text_features':
                        new_file.copy(o2, new_group, name=n2)
    
    #step 2
    with h5py.File(output_file, 'r+') as f:
        current_vid = None
        datasets = []
        groups = []
        start = True
        video = None
        p = False
        weight = 0
        
    
        for name, obj in f.items():
            #check if the object is a group or a dataset (usefull for repeat sweeps)
            if isinstance(obj, h5py.Dataset):
                continue
    
            #step 1: get the name of the group, and thus, the video
            video = name.split('_')[1:-1]
            
            if len(video) > 1:
                video = ['_'.join(video)]
            video = video[0]
            
            
            #step 1.1: if this group is a new video, aggregate the datasets and remove the useless groups
            #step 1.2: Set the current video as the new current vid
            if video != current_vid: 
                if start == False:
                    average_vector = np.mean([np.array(vec) for vec in datasets], axis=0)
                    weight = len(datasets)
                    
                    if current_vid not in f: 
                        dataset = f.create_dataset(current_vid, data=average_vector)
                        dataset.attrs['weight'] = weight
                    
                    else:
                        dataset = f[current_vid]
                        weights = [dataset.attrs.get('weight'), weight]
                        new_average = np.average(np.array([dataset[:], average_vector]), axis=0, weights=weights)
                        dataset[...] = new_average
                        dataset.attrs['weight'] = sum(weights)
    
                    
                        
                    #deleting redundant groups containing the old data (which is now a duplicate)
                    for group in groups:
                        del f[group]
                    
                    datasets = []
                    groups = []
                else: start = False
                current_vid = video
                
              
                
            
            #step 2: add the data and group to the variable
            for title, value in obj.items():
                datasets.append(value[:])
            groups.append(name)
    
    
        average_vector = np.mean([np.array(vec) for vec in datasets], axis=0)
        weight = len(datasets)
        
        if current_vid not in f: 
            dataset = f.create_dataset(current_vid, data=average_vector)
            dataset.attrs['weight'] = weight
        
        else:
            dataset = f[current_vid]
            weights = [dataset.attrs.get('weight'), weight]
            new_average = np.average(np.array([dataset[:], average_vector]), axis=0, weights=weights)
            dataset[...] = new_average
            dataset.attrs['weight'] = sum(weights)
    
        for group in groups:
                del f[group]



        

In [146]:
import netflix_recommender as nr

nr.restructure_features('../../datacsv/features.h5', '../../datacsv/shortend_features.h5')

In [148]:
def print_structure(name, obj):
    """Callback function to print the layout of the file"""
    if isinstance(obj, h5py.Dataset):
        print(f"Dataset: {name} (Shape: {obj.shape}, Type: {obj.dtype})")
    elif isinstance(obj, h5py.Group):
        print(f"Group: {name}")

# Open the file
with h5py.File('../../datacsv/shortend_features.h5', 'r') as f:
    # This walks through the entire file hierarchy
    f.visititems(print_structure)

Dataset: --Jiv5iYqT8 (Shape: (768,), Type: float32)
Dataset: -DXU2ZHuiTs (Shape: (768,), Type: float32)
Dataset: -JhNO_E3aEE (Shape: (768,), Type: float32)
Dataset: -N2mhlvygq0 (Shape: (768,), Type: float32)
Dataset: -PzL4MTu3to (Shape: (768,), Type: float32)
Dataset: -ZTTZtzzbmI (Shape: (768,), Type: float32)
Dataset: -ZlL0LLfjKY (Shape: (768,), Type: float32)
Dataset: -dMBMU9FCQU (Shape: (768,), Type: float32)
Dataset: -dkz4Sk6UuA (Shape: (768,), Type: float32)
Dataset: -qSADHn4nqw (Shape: (768,), Type: float32)
Dataset: -r_jjQ_idz8 (Shape: (768,), Type: float32)
Dataset: -yyoLJuNIJU (Shape: (768,), Type: float32)
Dataset: 0AspXDFcGlw (Shape: (768,), Type: float32)
Dataset: 0DFBoLZC3Bw (Shape: (768,), Type: float32)
Dataset: 0IyuK069I-w (Shape: (768,), Type: float32)
Dataset: 0OqXqd_s0ho (Shape: (768,), Type: float32)
Dataset: 0X0-NpZpx6U (Shape: (768,), Type: float32)
Dataset: 0XOzPjsD_ec (Shape: (768,), Type: float32)
Dataset: 0_avQPM3L90 (Shape: (768,), Type: float32)
Dataset: 0fw